# 📊 Análise Exploratória de Dados (EDA) - Indicadores Educacionais (INEP / MEC)
## Pipeline ETL com Arquitetura Medalhão (Camada Prata & Ouro)
---
Este notebook documenta formalmente a resposta a **5 Perguntas Estratégicas de Negócio** com base no Censo Escolar da Educação Básica do INEP, utilizando manipulação analítica em Pandas, estatística descritiva e matrizes de correlação.

In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

# Localização robusta dos caminhos relativos da Camada Prata
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'eda' else Path.cwd()
SILVER_FILE = BASE_DIR / 'data' / 'silver' / 'indicadores_educacionais_silver.parquet'
if not SILVER_FILE.exists():
    SILVER_FILE = BASE_DIR / 'data' / 'silver' / 'indicadores_educacionais_silver.csv'

df = pd.read_parquet(SILVER_FILE) if str(SILVER_FILE).endswith('.parquet') else pd.read_csv(SILVER_FILE)
print(f"Base Prata Carregada com Sucesso: {df.shape[0]} linhas x {df.shape[1]} colunas.")

Base Prata Carregada com Sucesso: 1710 linhas x 220 colunas.


### 1. Pergunta Estratégica 1: Evolução Histórica da Taxa de Aprovação no Ensino Médio por Região

In [ ]:
df_total = df[(df['localizacao'] == 'Total') & (df['rede'] == 'Total') & (df['ano'] >= 2013)]
fig1 = px.line(
    df_total,
    x='ano',
    y='taxa_aprovacao_em',
    color='regiao',
    markers=True,
    title='Evolução da Taxa de Aprovação no Ensino Médio por Região (2013-2024)',
    labels={'ano': 'Ano Letivo', 'taxa_aprovacao_em': 'Taxa de Aprovação (%)', 'regiao': 'Macrorregião'},
    template='plotly_white'
)
fig1.show()

### 2. Pergunta Estratégica 2: Disparidade de Abandono Escolar entre Rede Pública e Privada

In [ ]:
ano_max = df['ano'].max()
df_abandono = df[(df['ano'] == ano_max) & (df['localizacao'] == 'Total') & (df['rede'].isin(['Pública', 'Privada']))]
fig2 = px.bar(
    df_abandono,
    x='regiao',
    y='taxa_abandono_em',
    color='rede',
    barmode='group',
    title=f'Taxa de Abandono Escolar no Ensino Médio: Pública vs Privada ({ano_max})',
    labels={'taxa_abandono_em': 'Taxa de Abandono (%)', 'regiao': 'Região', 'rede': 'Dependência Administrativa'},
    template='plotly_white'
)
fig2.show()

### 3. Pergunta Estratégica 3: Distorção Idade-Série (TDI) e Desigualdade Territorial (Urbana x Rural)

In [ ]:
df_tdi = df[(df['ano'] == ano_max) & (df['rede'] == 'Total') & (df['localizacao'].isin(['Urbana', 'Rural']))]
fig3 = px.bar(
    df_tdi,
    x='regiao',
    y='tdi_ef',
    color='localizacao',
    barmode='group',
    title=f'Distorção Idade-Série no Ensino Fundamental por Localização ({ano_max})',
    labels={'tdi_ef': 'TDI Fundamental (%)', 'regiao': 'Região', 'localizacao': 'Zona Territorial'},
    template='plotly_white'
)
fig3.show()

### 4. Pergunta Estratégica 4: Formação Docente Superior (DSU) vs Desempenho Escolar

In [ ]:
df_dsu_apr = df[(df['localizacao'] == 'Total') & (df['rede'] == 'Total') & (df['ano'] >= 2018)]
fig4 = px.scatter(
    df_dsu_apr,
    x='dsu_em',
    y='taxa_aprovacao_em',
    color='regiao',
    size='atu_em',
    hover_data=['ano'],
    title='Relação entre Formação Docente (DSU) e Taxa de Aprovação no Ensino Médio',
    labels={'dsu_em': 'Docentes c/ Ensino Superior (%)', 'taxa_aprovacao_em': 'Taxa de Aprovação (%)'},
    template='plotly_white'
)
fig4.show()

### 5. Pergunta Estratégica 5: Carga Horária (HAD) e Efeito Protetor da Escola em Tempo Integral

In [ ]:
df_had = df[(df['ano'] == ano_max) & (df['localizacao'] == 'Total') & (df['rede'] == 'Total')]
fig5 = px.bar(
    df_had.sort_values('had_em', ascending=False),
    x='regiao',
    y='had_em',
    text='had_em',
    title=f'Ranking de Horas-Aula Diária (HAD) no Ensino Médio por Região ({ano_max})',
    labels={'had_em': 'Horas-Aula Diária (horas)', 'regiao': 'Região'},
    template='plotly_white'
)
fig5.update_traces(texttemplate='%{text:.1f}h', textposition='outside')
fig5.show()

### 6. Matriz de Correlação Linear de Pearson

In [ ]:
cols_corr = ['taxa_aprovacao_em', 'taxa_abandono_em', 'tdi_em', 'had_em', 'dsu_em', 'atu_em']
corr_matrix = df_total[cols_corr].corr().round(2)
fig_corr = px.imshow(
    corr_matrix,
    text_auto=True,
    aspect='auto',
    color_continuous_scale='RdBu_r',
    title='Matriz de Correlação de Pearson - Indicadores Educacionais do Ensino Médio'
)
fig_corr.show()